In [42]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
import os
import sys

import cv2

import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F

from box import Box
import yaml

import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation as R
import random 

import time 

root_dir = os.path.abspath('../..')
if root_dir not in sys.path:
    sys.path.append(root_dir)

from src.models.update import Update, neighbours
from src.models.graph_inference import Graph as Graph_interference
from src.models.graph_training import Graph as Graph_train
from src.models.bundle_adjustment import BundleAdjustment
from src.models.dpso import DPSO 

In [44]:
model_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/model.yaml'
sonar_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/sonar.yaml'
train_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/training.yaml'

# with open(model_config_pth, "r") as f:
#     model_config = Box(yaml.safe_load(f))

# with open(sonar_config_pth, "r") as f:
#     sonar_config = Box(yaml.safe_load(f))

# with open(train_config_pth, "r") as f:
#     train_config = Box(yaml.safe_load(f))


# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# init graph instance
# PatchGraph_i = Graph_interference(model_config, sonar_config)
# PatchGraph_t = Graph_train(model_config, sonar_config, train_config)

# # init update operater instance
# UpdateOperator_t = Update(model_config)
# UpdateOperator_i = Update(model_config)



In [45]:
# test data 
start_num = 200
data_pth_root = f'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/seq_1/fls' # 200.png'

tmp_img = cv2.imread(os.path.join(data_pth_root, f'0.png'), 0)
h, w = tmp_img.shape

frames_in_series = 5
batch_size = 3
t_0 = 1.0
dt = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

time_tensor = torch.tensor([t_0 + i*dt for i in range(frames_in_series)], device = device, dtype = torch.float)
time_tensor = time_tensor.unsqueeze(0)
time_tensor = torch.cat([time_tensor for _ in range(batch_size)], dim = 0)

# read frame 

frame_sim_b = torch.zeros((batch_size, frames_in_series, 1, h, w))

for j in range(batch_size):
    for i in range(frames_in_series):
        pth = os.path.join(data_pth_root, f'{start_num+batch_size*j+i}.png')
        frame_sim = cv2.imread(pth, 0)
        frame_sim = frame_sim.astype(np.uint8)
        
        frame_sim_b[j, i, :, :] = torch.from_numpy(frame_sim).unsqueeze(0).float()


print('-'*80)
print(f'Input data format:')
print(f'simulated tensor shape: {frame_sim.shape}, data type: {frame_sim.dtype}')
print(f'simulated tensors batch shape: {frame_sim_b.shape}, data type: {frame_sim_b.dtype}')
print(f'time tensor: {time_tensor.shape}')
print('-'*80)

--------------------------------------------------------------------------------
Input data format:
simulated tensor shape: (800, 768), data type: uint8
simulated tensors batch shape: torch.Size([3, 5, 1, 800, 768]), data type: torch.float32
time tensor: torch.Size([3, 5])
--------------------------------------------------------------------------------


# **Training test**

In [46]:
dpso_t = DPSO(model_config_pth, sonar_config_pth, train_config_pth, mode = 'train', output_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/test')

## forward:

In [47]:
# output = dpso_t(frame_sim_b, time_tensor)

In [48]:
# print(output.shape)


## backward:

In [52]:
optimizer = torch.optim.Adam(dpso_t.parameters(), lr=1e-4)
optimizer.zero_grad()

pred = dpso_t(frame_sim_b, time_tensor)

gt = torch.zeros_like(pred.clone())

criterion = nn.MSELoss()
loss = criterion(pred, gt)

print(f'loss: {loss}')

loss.backward()

optimizer.step()

loss: 0.1439562439918518


In [54]:
print(pred)

tensor([[[-8.5971e-03,  1.5177e-01,  8.9234e-03,  1.4889e-03, -2.1948e-01,
           6.0721e-03,  9.7560e-01],
         [-8.4492e-03, -6.1675e-02, -9.9513e-03,  7.4121e-03,  1.4730e-01,
          -2.6093e-03,  9.8906e-01],
         [ 2.4888e-03, -8.8038e-02, -1.0173e-02, -2.3189e-03,  9.0074e-02,
          -1.0910e-03,  9.9593e-01],
         [-2.9998e-03,  1.1951e-01,  2.1167e-03,  7.5359e-03,  9.7019e-02,
           4.3622e-03,  9.9525e-01],
         [ 2.6291e-03, -2.0431e-01,  3.8522e-03, -1.3434e-02, -1.8501e-02,
          -3.8840e-03,  9.9973e-01]],

        [[-1.8745e-04, -6.3097e-03, -8.6825e-04,  1.3222e-03,  9.3401e-02,
          -1.5884e-04,  9.9563e-01],
         [-4.9636e-03,  4.2149e-02, -3.3392e-02,  8.2319e-03,  1.4699e-02,
          -3.5100e-04,  9.9986e-01],
         [ 4.2840e-03, -3.3674e-02,  3.2181e-02, -1.0065e-02, -2.0346e-02,
          -3.9444e-04,  9.9974e-01],
         [-3.4018e-03, -6.9756e-02, -2.0617e-02,  1.4311e-02,  1.5313e-01,
          -7.6481e-04,  9.8

# **Inference test**

In [50]:
dpso_i = DPSO(model_config_pth, sonar_config_pth, train_config_pth, mode = 'inference', output_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/test')

dpso_i.eval()

DPSO(
  (PatchGraph): Graph(
    (patchifier): Patchifier(
      (feature_extractor): Encoder(
        (conv1): Conv2d(1, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
        (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (ResBlock1): Sequential(
          (0): ResidualBlock(
            (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
            (norm2): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
          )
          (1): ResidualBlock(
            (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (conv2): Conv2d(32, 32, kernel_size=(3, 3), str

## forward:

In [51]:
b, n, c, h, w = frame_sim_b.shape
print(f'Batch size: {b}, frames per batch: {n}')

prev_t = 0
# with torch.inference_mode(): # DO NOT USE THIS! this mode doesn't allow to turn grad on and it kills bundle adjustment 
with torch.no_grad():
    for batch in range(b):
        for frame_num in range(n):


            frame = frame_sim_b[batch, frame_num, :, :, :].unsqueeze(0).unsqueeze(0)
            t = time_tensor[batch, frame_num] + prev_t


            pos, time, nu = dpso_i(frame, t)
            print(f't: {time}, n: {nu}, pos: {pos}')
        prev_t = t

    dpso_i.close()

Batch size: 3, frames per batch: 5
t: 1.0, n: 1, pos: tensor([0., 0., 0., 0., 0., 0., 1.])
t: 1.5, n: 2, pos: tensor([-3.7333e-03,  3.3439e-01, -2.6059e-03,  3.7820e-03, -2.7834e-01,
         7.1679e-04,  9.6047e-01])
t: 2.0, n: 3, pos: tensor([ 0.0426, -0.9385,  0.0476,  0.3033,  0.2818,  0.2851,  0.8645])
t: 2.5, n: 4, pos: tensor([ 0.0426, -0.9385,  0.0476,  0.3033,  0.2818,  0.2851,  0.8645])
t: 3.0, n: 5, pos: tensor([ 0.0426, -0.9385,  0.0476,  0.3033,  0.2818,  0.2851,  0.8645])
t: 4.0, n: 6, pos: tensor([ 0.0426, -0.9385,  0.0476,  0.3033,  0.2818,  0.2851,  0.8645])
t: 4.5, n: 7, pos: tensor([ 0.0426, -0.9385,  0.0476,  0.3033,  0.2818,  0.2851,  0.8645])
t: 5.0, n: 8, pos: tensor([ 0.0426, -0.9385,  0.0476,  0.3033,  0.2818,  0.2851,  0.8645])
t: 5.5, n: 9, pos: tensor([ 0.0426, -0.9385,  0.0476,  0.3033,  0.2818,  0.2851,  0.8645])
t: 6.0, n: 10, pos: tensor([ 0.0426, -0.9385,  0.0476,  0.3033,  0.2818,  0.2851,  0.8645])
t: 7.0, n: 11, pos: tensor([-0.9021, -0.2243,  0.0866